In [3]:
#imports
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt

from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm

import dask
from dask_cuda import LocalCUDACluster
from dask.distributed import Client

2026-01-21 16:13:01.160159: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import tensorflow as tf

def check_tensorflow_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    print(f"Detected GPUs: {gpus}")
    if gpus:
        for gpu in gpus:
            print(f"Device: {gpu}")
            tf.config.experimental.set_memory_growth(gpu, True)

check_tensorflow_gpu()

Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Device: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [5]:
def setup_tensorflow():
    """Ensure TensorFlow uses GPU and configures memory growth."""
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    return gpus

def load_and_preprocess_data(data_root):
    """Load, preprocess the data and return processed data object."""
    seelig = scm.scMPRA_data.from_tsv(f"{data_root}/seelig/seelig_counts_grouped.txt")
    seelig.set_negative_controls(["AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
    "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"])
    seelig.set_reference_cell("HEPG2")
    seelig.ortho_filter()
    return seelig

def fit_tensorzinb(seelig, client):
    """ Fit model using preprocessed data in a distributed manner. """
    primordial = scm.ortho()
    by_cre, by_cre_design = scm.standard_fit(client=client,
                                             data=seelig,
                                             split="cre_id")
    return by_cre, by_cre_design

In [ ]:
# Initialize Dask Cluster
cluster = LocalCUDACluster(protocol='ucx', enable_nvlink=True)
client = Client(cluster)

data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"

seelig = load_and_preprocess_data(data_root)
primordial = scm.ortho()

by_cre, by_cre_design = scm.standard_fit(client=client,
                                             data=seelig,
                                             split="cre_id")

# Confirm TensorFlow sees GPUs within Dask Workers
gpu_info = client.run(setup_tensorflow)
print(f"GPU Info from Dask Workers: {gpu_info}")

scMPRAforge: INFO: Dropped 81 of 2688 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [7]:
client.close()
cluster.close()

In [9]:
#load data
seelig=scm.scMPRA_data.from_tsv(f"{data_root}/seelig/seelig_counts_grouped.txt")
seelig.set_negative_controls(["AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
    "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"])
seelig.set_reference_cell("HEPG2")
seelig.ortho_filter()

primordial=scm.ortho()


scMPRAforge: INFO: Dropped 81 of 2688 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [10]:
by_cre, by_cre_design = scm.standard_fit(client=client,
                                             data=seelig,
                                             split="cre_id")

In [12]:
def check_gpu_usage():
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    gpu_util_info = {}

    for gpu in gpus:
        mem_info = tf.config.experimental.get_memory_info('GPU:0')
        gpu_util_info[gpu.name] = {
            'current_memory': mem_info['current'],
            'peak_memory': mem_info['peak']
        }

    return gpu_util_info

# Run this function to check GPU utilization on each worker
gpu_utilization = client.run(check_gpu_usage)
print(f"Worker GPU Utilization: {gpu_utilization}")

Worker GPU Utilization: {'ucx://127.0.0.1:57883': {'/physical_device:GPU:0': {'current_memory': 0, 'peak_memory': 0}}}


In [ ]:
# primordial.criss_cross(client=client,
#                     dat=shendure)
# primordial.extract_params(client)
# primordial.save(path,name)

In [ ]:
t = #some cell type or cre
t_futures = client.submit(
            _smart_matrix,
            data=data[data[split]==t],
            split=split)

# mom init for ct models

In [ ]:
client.submit(
                _tensorzinb_fit,
                t_futures,
                t,
                init_method=init_method,
                init_vals=init_vals[t]